In [0]:
from pyspark.sql.functions import col, to_date, current_date, months_between, floor, concat_ws
from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
import mlflow
import mlflow.spark
import pandas as pd
import matplotlib.pyplot as plt
import os

In [0]:
# Load data
results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True, inferSchema=True)
drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True, inferSchema=True)
races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True, inferSchema=True)
circuits = spark.read.csv("/Volumes/gr5069/raw/f1_data/circuits.csv", header=True, inferSchema=True)

In [0]:
# 2. Prepare drivers
drivers = drivers.withColumn("dob", to_date(col("dob"), "yyyy-MM-dd"))

drivers = drivers.withColumn(
    "age",
    floor(months_between(current_date(), col("dob")) / 12)
)

drivers = drivers.withColumn(
    "driver_name",
    concat_ws(" ", col("forename"), col("surname"))
)

drivers_selected = drivers.select("driverId", "driver_name", "age")

# 3. Prepare races
races_selected = races.select("raceId", "year", "round", "circuitId")

# 4. Prepare results
results_selected = results.select(
    "raceId",
    "driverId",
    "constructorId",
    "grid",
    "positionOrder"
)

# 5. Prepare circuits
circuits_selected = circuits.select("circuitId", "country")

# 6. Join tables
df = results_selected \
    .join(drivers_selected, on="driverId", how="left") \
    .join(races_selected, on="raceId", how="left") \
    .join(circuits_selected, on="circuitId", how="left")

# 7. Clean data
df = df.withColumn("positionOrder", col("positionOrder").cast("double"))
df = df.withColumn("grid", col("grid").cast("double"))
df = df.withColumn("age", col("age").cast("double"))
df = df.withColumn("year", col("year").cast("double"))
df = df.withColumn("round", col("round").cast("double"))

df = df.dropna()

# 8. Final modeling table
df_model = df.select(
    "positionOrder",
    "grid",
    "age",
    "year",
    "round",
    "driverId",
    "constructorId",
    "circuitId",
    "country"
)

display(df_model.limit(5))
df_model.printSchema()
print(df_model.count())

In [0]:
# Convert categorical variables into numeric indices

# Each unique driver is assigned a numerical index
driver_indexer = StringIndexer(
    inputCol="driverId",
    outputCol="driverId_index",
    handleInvalid="keep"
)

# constructorId → constructorId_index
constructor_indexer = StringIndexer(
    inputCol="constructorId",
    outputCol="constructorId_index",
    handleInvalid="keep"
)

# circuitId → circuitId_index
circuit_indexer = StringIndexer(
    inputCol="circuitId",
    outputCol="circuitId_index",
    handleInvalid="keep"
)

# country → country_index
country_indexer = StringIndexer(
    inputCol="country",
    outputCol="country_index",
    handleInvalid="keep"
)

In [0]:
# Define feature columns
feature_cols = [
    "grid",
    "age",
    "year",
    "round",
]

# Combine all features into a single vector
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

df_rf_features = assembler.transform(df_model).select("features", "positionOrder")

display(df_rf_features.limit(5))
df_rf_features.printSchema()

In [0]:
# Split data into training and testing sets
train_df, test_df = df_rf_features.randomSplit([0.8, 0.2], seed=42)

# Check the size of each split
print("Train count:", train_df.count())
print("Test count:", test_df.count())

In [0]:
# Define the model
# I use a Random Forest Regressor to predict the final race position (positionOrder).
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="positionOrder",
    numTrees=5,
    maxDepth=2,
    maxBins=8,
    seed=42
)

# Train the model
rf_model = rf.fit(train_df)

# Generate Prediction
predictions = rf_model.transform(test_df)

display(predictions.limit(10))

In [0]:
# Evaluate model performance
# Use four standard regression metrics: RMSE, MAE, MSE, and R2

rmse_evaluator = RegressionEvaluator(
    labelCol="positionOrder",
    predictionCol="prediction",
    metricName="rmse"
)

mae_evaluator = RegressionEvaluator(
    labelCol="positionOrder",
    predictionCol="prediction",
    metricName="mae"
)

mse_evaluator = RegressionEvaluator(
    labelCol="positionOrder",
    predictionCol="prediction",
    metricName="mse"
)

r2_evaluator = RegressionEvaluator(
    labelCol="positionOrder",
    predictionCol="prediction",
    metricName="r2"
)

rmse = rmse_evaluator.evaluate(predictions)
mae = mae_evaluator.evaluate(predictions)
mse = mse_evaluator.evaluate(predictions)
r2 = r2_evaluator.evaluate(predictions)

In [0]:
# Set MLflow experiment
mlflow.set_experiment("/Users/yt2968@columbia.edu/f1_race_position")

In [0]:
with mlflow.start_run(run_name="baseline_run") as run:

    # Train model
    rf = RandomForestRegressor(
        featuresCol="features",
        labelCol="positionOrder",
        numTrees=5,
        maxDepth=2,
        maxBins=8,
        seed=42
    )

    rf_model = rf.fit(train_df)
    predictions = rf_model.transform(test_df)

    # Evaluate metrics
    rmse = rmse_evaluator.evaluate(predictions)
    mae = mae_evaluator.evaluate(predictions)
    mse = mse_evaluator.evaluate(predictions)
    r2 = r2_evaluator.evaluate(predictions)

    # Log parameters
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("numTrees", 5)
    mlflow.log_param("maxDepth", 2)
    mlflow.log_param("maxBins", 8)

    # Log metrics
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2", r2)

    # Artifact 1: CSV
    os.makedirs("/tmp/f1_artifacts", exist_ok=True)
    pred_pd = predictions.select("positionOrder", "prediction").toPandas()
    pred_csv_path = "/tmp/f1_artifacts/predictions_run_baseline.csv"
    pred_pd.to_csv(pred_csv_path, index=False)
    mlflow.log_artifact(pred_csv_path)

    # Artifact 2: Plot
    plt.figure(figsize=(6,4))
    plt.scatter(pred_pd["positionOrder"], pred_pd["prediction"], alpha=0.5)
    plt.xlabel("Actual Position")
    plt.ylabel("Predicted Position")
    plt.title("Actual vs Predicted")
    plot_path = "/tmp/f1_artifacts/plot_run_baseline.png"
    plt.savefig(plot_path)
    plt.close()
    mlflow.log_artifact(plot_path)

    # Print run ID and experiment ID
    runID = run.info.run_id
    experimentID = run.info.experiment_id

    print(f"Run ID: {runID}")
    print(f"Experiment ID: {experimentID}")

In [0]:
#Second run
with mlflow.start_run(run_name="second_run") as run:

    # Train model
    rf = RandomForestRegressor(
        featuresCol="features",
        labelCol="positionOrder",
        numTrees=3,
        maxDepth=2,
        maxBins=8,
        seed=42
    )

    rf_model = rf.fit(train_df)
    predictions = rf_model.transform(test_df)

    # Evaluate metrics
    rmse = rmse_evaluator.evaluate(predictions)
    mae = mae_evaluator.evaluate(predictions)
    mse = mse_evaluator.evaluate(predictions)
    r2 = r2_evaluator.evaluate(predictions)

    # Log parameters
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("numTrees", 3)
    mlflow.log_param("maxDepth", 2)
    mlflow.log_param("maxBins", 8)

    # Log metrics
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2", r2)

    # Artifact 1: CSV
    os.makedirs("/tmp/f1_artifacts", exist_ok=True)
    pred_pd = predictions.select("positionOrder", "prediction").toPandas()
    pred_csv_path = "/tmp/f1_artifacts/predictions_run_second.csv"
    pred_pd.to_csv(pred_csv_path, index=False)
    mlflow.log_artifact(pred_csv_path)

    # Artifact 2: Plot
    plt.figure(figsize=(6,4))
    plt.scatter(pred_pd["positionOrder"], pred_pd["prediction"], alpha=0.5)
    plt.xlabel("Actual Position")
    plt.ylabel("Predicted Position")
    plt.title("Actual vs Predicted")
    plot_path = "/tmp/f1_artifacts/plot_run_second.png"
    plt.savefig(plot_path)
    plt.close()
    mlflow.log_artifact(plot_path)

    # Print run ID and experiment ID
    runID = run.info.run_id
    experimentID = run.info.experiment_id

    print(f"Run ID: {runID}")
    print(f"Experiment ID: {experimentID}")

In [0]:
#Third run
with mlflow.start_run(run_name="third_run") as run:

    # Train model
    rf = RandomForestRegressor(
        featuresCol="features",
        labelCol="positionOrder",
        numTrees=8,
        maxDepth=2,
        maxBins=8,
        seed=42
    )

    rf_model = rf.fit(train_df)
    predictions = rf_model.transform(test_df)

    # Evaluate metrics
    rmse = rmse_evaluator.evaluate(predictions)
    mae = mae_evaluator.evaluate(predictions)
    mse = mse_evaluator.evaluate(predictions)
    r2 = r2_evaluator.evaluate(predictions)

    # Log parameters
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("numTrees", 8)
    mlflow.log_param("maxDepth", 2)
    mlflow.log_param("maxBins", 8)

    # Log metrics
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2", r2)

    # Artifact 1: CSV
    os.makedirs("/tmp/f1_artifacts", exist_ok=True)
    pred_pd = predictions.select("positionOrder", "prediction").toPandas()
    pred_csv_path = "/tmp/f1_artifacts/predictions_run_third.csv"
    pred_pd.to_csv(pred_csv_path, index=False)
    mlflow.log_artifact(pred_csv_path)

    # Artifact 2: Plot
    plt.figure(figsize=(6,4))
    plt.scatter(pred_pd["positionOrder"], pred_pd["prediction"], alpha=0.5)
    plt.xlabel("Actual Position")
    plt.ylabel("Predicted Position")
    plt.title("Actual vs Predicted")
    plot_path = "/tmp/f1_artifacts/plot_run_third.png"
    plt.savefig(plot_path)
    plt.close()
    mlflow.log_artifact(plot_path)

    # Print run ID and experiment ID
    runID = run.info.run_id
    experimentID = run.info.experiment_id

    print(f"Run ID: {runID}")
    print(f"Experiment ID: {experimentID}")

In [0]:
#Fourth run
with mlflow.start_run(run_name="fourth_run") as run:

    # Train model
    rf = RandomForestRegressor(
        featuresCol="features",
        labelCol="positionOrder",
        numTrees=5,
        maxDepth=3,
        maxBins=8,
        seed=42
    )

    rf_model = rf.fit(train_df)
    predictions = rf_model.transform(test_df)

    # Evaluate metrics
    rmse = rmse_evaluator.evaluate(predictions)
    mae = mae_evaluator.evaluate(predictions)
    mse = mse_evaluator.evaluate(predictions)
    r2 = r2_evaluator.evaluate(predictions)

    # Log parameters
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("numTrees", 5)
    mlflow.log_param("maxDepth", 3)
    mlflow.log_param("maxBins", 8)

    # Log metrics
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2", r2)

    # Artifact 1: CSV
    os.makedirs("/tmp/f1_artifacts", exist_ok=True)
    pred_pd = predictions.select("positionOrder", "prediction").toPandas()
    pred_csv_path = "/tmp/f1_artifacts/predictions_run_fourth.csv"
    pred_pd.to_csv(pred_csv_path, index=False)
    mlflow.log_artifact(pred_csv_path)

    # Artifact 2: Plot
    plt.figure(figsize=(6,4))
    plt.scatter(pred_pd["positionOrder"], pred_pd["prediction"], alpha=0.5)
    plt.xlabel("Actual Position")
    plt.ylabel("Predicted Position")
    plt.title("Actual vs Predicted")
    plot_path = "/tmp/f1_artifacts/plot_run_fourth.png"
    plt.savefig(plot_path)
    plt.close()
    mlflow.log_artifact(plot_path)

    # Print run ID and experiment ID
    runID = run.info.run_id
    experimentID = run.info.experiment_id

    print(f"Run ID: {runID}")
    print(f"Experiment ID: {experimentID}")

In [0]:
#Fifth run
with mlflow.start_run(run_name="fifth_run") as run:

    # Train model
    rf = RandomForestRegressor(
        featuresCol="features",
        labelCol="positionOrder",
        numTrees=3,
        maxDepth=3,
        maxBins=8,
        seed=42
    )

    rf_model = rf.fit(train_df)
    predictions = rf_model.transform(test_df)

    # Evaluate metrics
    rmse = rmse_evaluator.evaluate(predictions)
    mae = mae_evaluator.evaluate(predictions)
    mse = mse_evaluator.evaluate(predictions)
    r2 = r2_evaluator.evaluate(predictions)

    # Log parameters
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("numTrees", 3)
    mlflow.log_param("maxDepth", 3)
    mlflow.log_param("maxBins", 8)

    # Log metrics
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2", r2)

    # Artifact 1: CSV
    os.makedirs("/tmp/f1_artifacts", exist_ok=True)
    pred_pd = predictions.select("positionOrder", "prediction").toPandas()
    pred_csv_path = "/tmp/f1_artifacts/predictions_run_fifth.csv"
    pred_pd.to_csv(pred_csv_path, index=False)
    mlflow.log_artifact(pred_csv_path)

    # Artifact 2: Plot
    plt.figure(figsize=(6,4))
    plt.scatter(pred_pd["positionOrder"], pred_pd["prediction"], alpha=0.5)
    plt.xlabel("Actual Position")
    plt.ylabel("Predicted Position")
    plt.title("Actual vs Predicted")
    plot_path = "/tmp/f1_artifacts/plot_run_fifth.png"
    plt.savefig(plot_path)
    plt.close()
    mlflow.log_artifact(plot_path)

    # Print run ID and experiment ID
    runID = run.info.run_id
    experimentID = run.info.experiment_id

    print(f"Run ID: {runID}")
    print(f"Experiment ID: {experimentID}")

In [0]:
#Sixth run
with mlflow.start_run(run_name="sixth_run") as run:

    # Train model
    rf = RandomForestRegressor(
        featuresCol="features",
        labelCol="positionOrder",
        numTrees=8,
        maxDepth=3,
        maxBins=8,
        seed=42
    )

    rf_model = rf.fit(train_df)
    predictions = rf_model.transform(test_df)

    # Evaluate metrics
    rmse = rmse_evaluator.evaluate(predictions)
    mae = mae_evaluator.evaluate(predictions)
    mse = mse_evaluator.evaluate(predictions)
    r2 = r2_evaluator.evaluate(predictions)

    # Log parameters
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("numTrees", 8)
    mlflow.log_param("maxDepth", 3)
    mlflow.log_param("maxBins", 8)

    # Log metrics
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2", r2)

    # Artifact 1: CSV
    os.makedirs("/tmp/f1_artifacts", exist_ok=True)
    pred_pd = predictions.select("positionOrder", "prediction").toPandas()
    pred_csv_path = "/tmp/f1_artifacts/predictions_run_sixth.csv"
    pred_pd.to_csv(pred_csv_path, index=False)
    mlflow.log_artifact(pred_csv_path)

    # Artifact 2: Plot
    plt.figure(figsize=(6,4))
    plt.scatter(pred_pd["positionOrder"], pred_pd["prediction"], alpha=0.5)
    plt.xlabel("Actual Position")
    plt.ylabel("Predicted Position")
    plt.title("Actual vs Predicted")
    plot_path = "/tmp/f1_artifacts/plot_run_sixth.png"
    plt.savefig(plot_path)
    plt.close()
    mlflow.log_artifact(plot_path)

    # Print run ID and experiment ID
    runID = run.info.run_id
    experimentID = run.info.experiment_id

    print(f"Run ID: {runID}")
    print(f"Experiment ID: {experimentID}")

In [0]:
#Seventh run
with mlflow.start_run(run_name="seventh_run") as run:

    # Train model
    rf = RandomForestRegressor(
        featuresCol="features",
        labelCol="positionOrder",
        numTrees=5,
        maxDepth=2,
        maxBins=10,
        seed=42
    )

    rf_model = rf.fit(train_df)
    predictions = rf_model.transform(test_df)

    # Evaluate metrics
    rmse = rmse_evaluator.evaluate(predictions)
    mae = mae_evaluator.evaluate(predictions)
    mse = mse_evaluator.evaluate(predictions)
    r2 = r2_evaluator.evaluate(predictions)

    # Log parameters
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("numTrees", 5)
    mlflow.log_param("maxDepth", 2)
    mlflow.log_param("maxBins", 10)

    # Log metrics
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2", r2)

    # Artifact 1: CSV
    os.makedirs("/tmp/f1_artifacts", exist_ok=True)
    pred_pd = predictions.select("positionOrder", "prediction").toPandas()
    pred_csv_path = "/tmp/f1_artifacts/predictions_run_seventh.csv"
    pred_pd.to_csv(pred_csv_path, index=False)
    mlflow.log_artifact(pred_csv_path)

    # Artifact 2: Plot
    plt.figure(figsize=(6,4))
    plt.scatter(pred_pd["positionOrder"], pred_pd["prediction"], alpha=0.5)
    plt.xlabel("Actual Position")
    plt.ylabel("Predicted Position")
    plt.title("Actual vs Predicted")
    plot_path = "/tmp/f1_artifacts/plot_run_seventh.png"
    plt.savefig(plot_path)
    plt.close()
    mlflow.log_artifact(plot_path)

    # Print run ID and experiment ID
    runID = run.info.run_id
    experimentID = run.info.experiment_id

    print(f"Run ID: {runID}")
    print(f"Experiment ID: {experimentID}")

In [0]:
#Eighth run
with mlflow.start_run(run_name="eighth_run") as run:

    # Train model
    rf = RandomForestRegressor(
        featuresCol="features",
        labelCol="positionOrder",
        numTrees=3,
        maxDepth=2,
        maxBins=10,
        seed=42
    )

    rf_model = rf.fit(train_df)
    predictions = rf_model.transform(test_df)

    # Evaluate metrics
    rmse = rmse_evaluator.evaluate(predictions)
    mae = mae_evaluator.evaluate(predictions)
    mse = mse_evaluator.evaluate(predictions)
    r2 = r2_evaluator.evaluate(predictions)

    # Log parameters
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("numTrees", 3)
    mlflow.log_param("maxDepth", 2)
    mlflow.log_param("maxBins", 10)

    # Log metrics
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2", r2)

    # Artifact 1: CSV
    os.makedirs("/tmp/f1_artifacts", exist_ok=True)
    pred_pd = predictions.select("positionOrder", "prediction").toPandas()
    pred_csv_path = "/tmp/f1_artifacts/predictions_run_eighth.csv"
    pred_pd.to_csv(pred_csv_path, index=False)
    mlflow.log_artifact(pred_csv_path)

    # Artifact 2: Plot
    plt.figure(figsize=(6,4))
    plt.scatter(pred_pd["positionOrder"], pred_pd["prediction"], alpha=0.5)
    plt.xlabel("Actual Position")
    plt.ylabel("Predicted Position")
    plt.title("Actual vs Predicted")
    plot_path = "/tmp/f1_artifacts/plot_run_eighth.png"
    plt.savefig(plot_path)
    plt.close()
    mlflow.log_artifact(plot_path)

    # Print run ID and experiment ID
    runID = run.info.run_id
    experimentID = run.info.experiment_id

    print(f"Run ID: {runID}")
    print(f"Experiment ID: {experimentID}")

In [0]:
#Ninth run
with mlflow.start_run(run_name="ninth_run") as run:

    # Train model
    rf = RandomForestRegressor(
        featuresCol="features",
        labelCol="positionOrder",
        numTrees=8,
        maxDepth=2,
        maxBins=10,
        seed=42
    )

    rf_model = rf.fit(train_df)
    predictions = rf_model.transform(test_df)

    # Evaluate metrics
    rmse = rmse_evaluator.evaluate(predictions)
    mae = mae_evaluator.evaluate(predictions)
    mse = mse_evaluator.evaluate(predictions)
    r2 = r2_evaluator.evaluate(predictions)

    # Log parameters
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("numTrees", 8)
    mlflow.log_param("maxDepth", 2)
    mlflow.log_param("maxBins", 10)

    # Log metrics
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2", r2)

    # Artifact 1: CSV
    os.makedirs("/tmp/f1_artifacts", exist_ok=True)
    pred_pd = predictions.select("positionOrder", "prediction").toPandas()
    pred_csv_path = "/tmp/f1_artifacts/predictions_run_ninth.csv"
    pred_pd.to_csv(pred_csv_path, index=False)
    mlflow.log_artifact(pred_csv_path)

    # Artifact 2: Plot
    plt.figure(figsize=(6,4))
    plt.scatter(pred_pd["positionOrder"], pred_pd["prediction"], alpha=0.5)
    plt.xlabel("Actual Position")
    plt.ylabel("Predicted Position")
    plt.title("Actual vs Predicted")
    plot_path = "/tmp/f1_artifacts/plot_run_ninth.png"
    plt.savefig(plot_path)
    plt.close()
    mlflow.log_artifact(plot_path)

    # Print run ID and experiment ID
    runID = run.info.run_id
    experimentID = run.info.experiment_id

    print(f"Run ID: {runID}")
    print(f"Experiment ID: {experimentID}")

In [0]:
#Tenth run
with mlflow.start_run(run_name="tenth_run") as run:

    # Train model
    rf = RandomForestRegressor(
        featuresCol="features",
        labelCol="positionOrder",
        numTrees=5,
        maxDepth=3,
        maxBins=10,
        seed=42
    )

    rf_model = rf.fit(train_df)
    predictions = rf_model.transform(test_df)

    # Evaluate metrics
    rmse = rmse_evaluator.evaluate(predictions)
    mae = mae_evaluator.evaluate(predictions)
    mse = mse_evaluator.evaluate(predictions)
    r2 = r2_evaluator.evaluate(predictions)

    # Log parameters
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("numTrees", 5)
    mlflow.log_param("maxDepth", 3)
    mlflow.log_param("maxBins", 10)

    # Log metrics
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2", r2)

    # Artifact 1: CSV
    os.makedirs("/tmp/f1_artifacts", exist_ok=True)
    pred_pd = predictions.select("positionOrder", "prediction").toPandas()
    pred_csv_path = "/tmp/f1_artifacts/predictions_run_tenth.csv"
    pred_pd.to_csv(pred_csv_path, index=False)
    mlflow.log_artifact(pred_csv_path)

    # Artifact 2: Plot
    plt.figure(figsize=(6,4))
    plt.scatter(pred_pd["positionOrder"], pred_pd["prediction"], alpha=0.5)
    plt.xlabel("Actual Position")
    plt.ylabel("Predicted Position")
    plt.title("Actual vs Predicted")
    plot_path = "/tmp/f1_artifacts/plot_run_tenth.png"
    plt.savefig(plot_path)
    plt.close()
    mlflow.log_artifact(plot_path)

    # Print run ID and experiment ID
    runID = run.info.run_id
    experimentID = run.info.experiment_id

    print(f"Run ID: {runID}")
    print(f"Experiment ID: {experimentID}")

I conducted 10 experiments using a Random Forest Regressor with different hyperparameter configurations, including variations in the number of trees, maximum tree depth, and number of bins.

The best-performing model was selected based on the lowest RMSE, as RMSE penalizes larger prediction errors more strongly and is a standard evaluation metric for regression tasks.

Among all runs, Run 10 achieved the best performance, with:

RMSE = 6.621
MAE = 5.496
MSE = 43.84
R² = 0.219

This model not only minimized prediction error but also achieved the highest R², indicating it explained the largest proportion of variance in race outcomes. Therefore, it provided the most accurate and reliable predictions of final race position among all tested configurations.
The improvement in performance suggests that increasing model complexity helps capture nonlinear relationships between features such as grid position, driver age, and race conditions.

Screenshots are under the folder "img"